# Fase 4: Análisis exhaustivo de compresión por componente y profundidad

Estudio sistemático del impacto de la compresión SVD selectiva sobre la calidad del modelo
y la estructura de embeddings. Se analizan dos dimensiones:

1. **Por componente**: Query, Key, Value, Attention Output, FFN Intermediate, FFN Output,
   Atención completa, FFN completo
2. **Por profundidad**: Capas tempranas (0-3), intermedias (4-7), tardías (8-11)

Cada configuración se evalúa a 3 niveles de rango SVD (256, 128, 64).

**Métricas**: F1 macro (GoEmotions test set) + Silhouette score (clusters t-SNE)

**Visualizaciones**: Scatter plots, contornos KDE, heatmaps comparativos

In [ ]:
import sys
import os

# En Colab, montar Drive y ajustar path al proyecto
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'

# En local
#PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from scipy.stats import gaussian_kde
from matplotlib.patches import Patch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.data.dataset import NUM_LABELS, EMOTION_NAMES
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    count_parameters,
    get_compression_ratio,
    get_target_layer_names,
    filter_layer_names,
)

sns.set_style('whitegrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Cargar modelo y datos

In [ ]:
model_path = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-final')

baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=NUM_LABELS,
    problem_type='multi_label_classification',
)
baseline_model.eval()
baseline_model.to(device)

params = count_parameters(baseline_model)
print(f'Parámetros totales: {params["total"]:,}')

In [ ]:
_, _, test_ds, emotion_names, data_collator = load_goemotions()
print(f'Test set: {len(test_ds)} ejemplos')

# Subsamplear test set para que t-SNE sea manejable
N_SAMPLES = 1500
rng = np.random.RandomState(42)
indices = rng.choice(len(test_ds), size=min(N_SAMPLES, len(test_ds)), replace=False)
indices.sort()
sub_test = test_ds.select(indices)
print(f'Subsample: {len(sub_test)} ejemplos')

In [ ]:
all_target_names = get_target_layer_names(baseline_model)
print(f'Capas comprimibles: {len(all_target_names)}')
for name in all_target_names[:6]:
    print(f'  {name}')
print('  ...')
for name in all_target_names[-2:]:
    print(f'  {name}')

## 2. Funciones auxiliares

In [ ]:
eval_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, 'results', 'tmp_eval'),
    per_device_eval_batch_size=64,
    fp16=(device == 'cuda'),
    report_to='none',
)


def evaluate_model(model):
    """Evalúa un modelo en el test set y devuelve el dict de métricas."""
    trainer = Trainer(
        model=model,
        args=eval_args,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    return trainer.evaluate(test_ds)

In [ ]:
@torch.no_grad()
def extract_cls_embeddings(model, dataset, batch_size=64):
    """Extrae embeddings [CLS] del último hidden state."""
    model.eval()
    model_device = next(model.parameters()).device

    all_embeddings = []
    all_labels = []

    for i in range(0, len(dataset), batch_size):
        batch_indices = range(i, min(i + batch_size, len(dataset)))
        batch = data_collator([dataset[j] for j in batch_indices])

        input_ids = batch['input_ids'].to(model_device)
        attention_mask = batch['attention_mask'].to(model_device)
        labels = batch['labels'].numpy()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        cls_emb = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        all_embeddings.append(cls_emb)
        all_labels.append(labels)

    return np.concatenate(all_embeddings, axis=0), np.concatenate(all_labels, axis=0)

In [ ]:
def run_joint_tsne(embeddings_dict, perplexity=30, n_iter=1000, seed=42):
    """Ejecuta t-SNE sobre embeddings concatenados de múltiples modelos.

    Returns:
        tsne_per_model: dict name -> coords (N, 2)
        bounds: tuple (x_min, x_max, y_min, y_max)
    """
    model_names = list(embeddings_dict.keys())
    all_embs = np.concatenate([embeddings_dict[n] for n in model_names], axis=0)
    n_per_model = len(embeddings_dict[model_names[0]])

    print(f'  Puntos totales: {all_embs.shape[0]} ({len(model_names)} modelos x {n_per_model})')

    tsne = TSNE(
        n_components=2, perplexity=perplexity, n_iter=n_iter,
        random_state=seed, init='pca', learning_rate='auto',
    )
    all_coords = tsne.fit_transform(all_embs)

    tsne_per_model = {}
    for i, name in enumerate(model_names):
        start = i * n_per_model
        end = start + n_per_model
        tsne_per_model[name] = all_coords[start:end]

    bounds = (
        all_coords[:, 0].min() - 2, all_coords[:, 0].max() + 2,
        all_coords[:, 1].min() - 2, all_coords[:, 1].max() + 2,
    )

    return tsne_per_model, bounds

In [ ]:
def plot_scatter_panel(tsne_per_model, bounds, dominant_emotion, top_indices,
                       f1_scores, ncols=3, title='', save_path=None):
    """Panel de scatter plots coloreados por emoción dominante."""
    model_names = list(tsne_per_model.keys())
    n_models = len(model_names)
    nrows = (n_models + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 6))
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes[np.newaxis, :]
    elif ncols == 1:
        axes = axes[:, np.newaxis]

    x_min, x_max, y_min, y_max = bounds
    cmap = plt.cm.tab10
    is_top = np.isin(dominant_emotion, top_indices)

    for idx, name in enumerate(model_names):
        row, col = divmod(idx, ncols)
        ax = axes[row, col]
        coords = tsne_per_model[name]

        other_mask = ~is_top
        ax.scatter(coords[other_mask, 0], coords[other_mask, 1],
                   c=[(0.85, 0.85, 0.85, 0.2)], s=3, rasterized=True)

        for rank_i, emo_idx in enumerate(top_indices):
            mask = dominant_emotion == emo_idx
            ax.scatter(coords[mask, 0], coords[mask, 1],
                       c=[cmap(rank_i)], s=8, alpha=0.6,
                       label=EMOTION_NAMES[emo_idx] if idx == 0 else None,
                       rasterized=True)

        f1 = f1_scores.get(name, 0)
        ax.set_title(f'{name}\nF1 macro = {f1:.4f}', fontsize=11, fontweight='bold')
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xticks([])
        ax.set_yticks([])

    for idx in range(n_models, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row, col].set_visible(False)

    axes[0, 0].legend(bbox_to_anchor=(-0.15, 0.5), loc='center right',
                      fontsize=8, frameon=True, framealpha=0.9, markerscale=2.5)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_kde_panel(tsne_per_model, bounds, dominant_emotion, top_indices,
                   f1_scores, ncols=3, n_kde_emotions=6, title='', save_path=None):
    """Panel de contornos KDE de densidad por emoción."""
    model_names = list(tsne_per_model.keys())
    n_models = len(model_names)
    nrows = (n_models + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 6))
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes[np.newaxis, :]
    elif ncols == 1:
        axes = axes[:, np.newaxis]

    x_min, x_max, y_min, y_max = bounds
    cmap = plt.cm.tab10
    top_for_kde = top_indices[:n_kde_emotions]

    for idx, name in enumerate(model_names):
        row, col = divmod(idx, ncols)
        ax = axes[row, col]
        coords = tsne_per_model[name]

        ax.scatter(coords[:, 0], coords[:, 1],
                   c=[(0.9, 0.9, 0.9, 0.15)], s=1, rasterized=True)

        for rank_i, emo_idx in enumerate(top_for_kde):
            mask = dominant_emotion == emo_idx
            if mask.sum() < 10:
                continue
            x = coords[mask, 0]
            y = coords[mask, 1]
            try:
                kde = gaussian_kde(np.vstack([x, y]), bw_method=0.3)
                xi = np.linspace(x_min, x_max, 100)
                yi = np.linspace(y_min, y_max, 100)
                xi, yi = np.meshgrid(xi, yi)
                zi = kde(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)
                color = cmap(rank_i)
                ax.contour(xi, yi, zi, levels=3, colors=[color],
                          linewidths=1.5, alpha=0.7)
                ax.contourf(xi, yi, zi, levels=3,
                           colors=[(*color[:3], 0.05), (*color[:3], 0.1),
                                   (*color[:3], 0.15), (*color[:3], 0.2)])
            except np.linalg.LinAlgError:
                pass

        f1 = f1_scores.get(name, 0)
        ax.set_title(f'{name}\nF1 macro = {f1:.4f}', fontsize=11, fontweight='bold')
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xticks([])
        ax.set_yticks([])

    for idx in range(n_models, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row, col].set_visible(False)

    legend_handles = [Patch(facecolor=cmap(i), alpha=0.5, label=EMOTION_NAMES[idx])
                      for i, idx in enumerate(top_for_kde)]
    axes[0, 0].legend(handles=legend_handles, bbox_to_anchor=(-0.15, 0.5),
                      loc='center right', fontsize=8, frameon=True, framealpha=0.9)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
def compute_silhouette_scores(tsne_per_model, dominant_emotion, top_indices):
    """Calcula silhouette score por modelo usando emociones top."""
    top_mask = np.isin(dominant_emotion, top_indices)
    top_labels = dominant_emotion[top_mask]
    scores = {}
    for name, coords in tsne_per_model.items():
        top_coords = coords[top_mask]
        sil = silhouette_score(
            top_coords, top_labels,
            sample_size=min(1000, len(top_coords)),
            random_state=42,
        )
        scores[name] = sil
    return scores

In [ ]:
def create_compressed_model(baseline, rank, component=None, layers=None):
    """Crea modelo comprimido con SVD selectivo.

    Aplica filter_layer_names + workaround para bug de ffn_output.
    """
    all_names = get_target_layer_names(baseline)
    target_names = filter_layer_names(all_names, component=component, layers=layers)

    # Workaround: ffn_output pattern 'output.dense' also matches attention.output.dense
    if component == 'ffn_output':
        target_names = [n for n in target_names if 'attention' not in n]

    compressed = apply_svd_compression(baseline, rank=rank, layer_names=target_names)
    return compressed, target_names

## 3. Evaluación del baseline

In [ ]:
# Evaluar baseline
baseline_metrics = evaluate_model(baseline_model)
baseline_f1 = baseline_metrics['eval_f1_macro']
print(f'Baseline F1 macro: {baseline_f1:.4f}')

# Extraer embeddings baseline
baseline_embs, baseline_labels = extract_cls_embeddings(baseline_model, sub_test)
print(f'Baseline embeddings: {baseline_embs.shape}')

# Emoción dominante y top 10
dominant_emotion = np.argmax(baseline_labels, axis=1)
unique, counts = np.unique(dominant_emotion, return_counts=True)
top_indices = unique[np.argsort(-counts)][:10]

print('\nTop 10 emociones dominantes:')
for idx in top_indices:
    count = counts[unique == idx][0]
    print(f'  {EMOTION_NAMES[idx]:>15s}: {count:>4d} ({count/len(dominant_emotion)*100:.1f}%)')

## 4. Estudio 1 — Compresión por componente

8 componentes × 3 rangos = 24 configuraciones.

| Componente | Descripción |
|---|---|
| Query (Q) | Matrices de query en self-attention |
| Key (K) | Matrices de key en self-attention |
| Value (V) | Matrices de value en self-attention |
| Attn Output | Proyección de salida de attention |
| FFN Inter | Capa intermedia del FFN (768→3072) |
| FFN Output | Capa de salida del FFN (3072→768) |
| Attention (all) | Q + K + V + Attn Output (4 matrices × 12 capas) |
| FFN (all) | FFN Inter + FFN Output (2 matrices × 12 capas) |

In [ ]:
COMPONENTS = [
    ('query', 'Query (Q)'),
    ('key', 'Key (K)'),
    ('value', 'Value (V)'),
    ('attention_output', 'Attn Output'),
    ('intermediate', 'FFN Inter'),
    ('ffn_output', 'FFN Output'),
    ('attention', 'Attention (all)'),
    ('ffn', 'FFN (all)'),
]

RANKS = [256, 128, 64]

print(f'Componentes: {len(COMPONENTS)}')
for comp_key, comp_label in COMPONENTS:
    names = filter_layer_names(all_target_names, component=comp_key)
    if comp_key == 'ffn_output':
        names = [n for n in names if 'attention' not in n]
    print(f'  {comp_label:<20s}: {len(names)} capas')
print(f'Rangos: {RANKS}')
print(f'Total modelos: {len(COMPONENTS) * len(RANKS)}')

In [ ]:
comp_results = []
comp_embeddings = {}

for comp_key, comp_label in COMPONENTS:
    for rank in RANKS:
        print(f'\n{"="*50}')
        print(f'Component: {comp_label}, Rank: {rank}')

        model, target_names = create_compressed_model(
            baseline_model, rank, component=comp_key,
        )
        model.to(device)

        metrics = evaluate_model(model)
        f1 = metrics['eval_f1_macro']
        ratio = get_compression_ratio(baseline_model, model)

        embs, _ = extract_cls_embeddings(model, sub_test)

        comp_results.append({
            'component': comp_label,
            'component_key': comp_key,
            'rank': rank,
            'f1_macro': f1,
            'compression_ratio': ratio,
            'n_layers': len(target_names),
        })
        comp_embeddings[(comp_key, rank)] = embs

        print(f'  F1 macro: {f1:.4f}')
        print(f'  Compression ratio: {ratio:.2f}x')
        print(f'  Layers compressed: {len(target_names)}')

        del model
        if device == 'cuda':
            torch.cuda.empty_cache()

print(f'\n{"="*50}')
print(f'Modelos evaluados: {len(comp_results)}')

### 4.1 t-SNE y visualización por componente

Para cada nivel de rango, ejecutamos t-SNE conjunto sobre baseline + 8 variantes de componente
(9 modelos × 1500 puntos = 13,500 puntos por run).

In [ ]:
comp_tsne = {}
comp_silhouette = {}

for rank in RANKS:
    print(f'\nt-SNE para rank {rank}...')

    emb_dict = {'Baseline': baseline_embs}
    for comp_key, comp_label in COMPONENTS:
        emb_dict[comp_label] = comp_embeddings[(comp_key, rank)]

    tsne_per_model, bounds = run_joint_tsne(emb_dict)
    comp_tsne[rank] = (tsne_per_model, bounds)

    sil_scores = compute_silhouette_scores(tsne_per_model, dominant_emotion, top_indices)
    for comp_key, comp_label in COMPONENTS:
        comp_silhouette[(comp_label, rank)] = sil_scores[comp_label]
    comp_silhouette[('Baseline', rank)] = sil_scores['Baseline']

    print(f'  Silhouette - Baseline: {sil_scores["Baseline"]:.4f}')
    for _, comp_label in COMPONENTS:
        print(f'  Silhouette - {comp_label}: {sil_scores[comp_label]:.4f}')

In [ ]:
results_dir = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(results_dir, exist_ok=True)

for rank in RANKS:
    tsne_per_model, bounds = comp_tsne[rank]

    f1_dict = {'Baseline': baseline_f1}
    for r in comp_results:
        if r['rank'] == rank:
            f1_dict[r['component']] = r['f1_macro']

    plot_scatter_panel(
        tsne_per_model, bounds, dominant_emotion, top_indices,
        f1_scores=f1_dict, ncols=3,
        title=f'Compresión por componente — Rank {rank}\n(scatter de emociones dominantes, t-SNE conjunto)',
        save_path=os.path.join(results_dir, f'comp_scatter_rank{rank}.png'),
    )

In [ ]:
for rank in RANKS:
    tsne_per_model, bounds = comp_tsne[rank]

    f1_dict = {'Baseline': baseline_f1}
    for r in comp_results:
        if r['rank'] == rank:
            f1_dict[r['component']] = r['f1_macro']

    plot_kde_panel(
        tsne_per_model, bounds, dominant_emotion, top_indices,
        f1_scores=f1_dict, ncols=3,
        title=f'Compresión por componente — Rank {rank}\n(densidad KDE de emociones, t-SNE conjunto)',
        save_path=os.path.join(results_dir, f'comp_kde_rank{rank}.png'),
    )

In [ ]:
print('Silhouette scores — Estudio por componente\n')

for rank in RANKS:
    print(f'{"="*45}')
    print(f'Rank {rank}')
    print(f'{"Modelo":<20s} | {"Silhouette":>10s}')
    print(f'{"-"*35}')
    print(f'{"Baseline":<20s} | {comp_silhouette[("Baseline", rank)]:>10.4f}')
    for _, comp_label in COMPONENTS:
        sil = comp_silhouette[(comp_label, rank)]
        print(f'{comp_label:<20s} | {sil:>10.4f}')
    print()

## 5. Estudio 2 — Compresión por profundidad

3 grupos de profundidad × 3 rangos = 9 configuraciones.
Se comprimen **todas** las matrices lineales (Q, K, V, Attn Out, FFN Inter, FFN Out)
dentro de cada grupo de capas.

| Grupo | Capas | Descripción |
|---|---|---|
| Early (0-3) | 0, 1, 2, 3 | Capas tempranas — features léxicas/sintácticas |
| Middle (4-7) | 4, 5, 6, 7 | Capas intermedias — features semánticas |
| Late (8-11) | 8, 9, 10, 11 | Capas tardías — features de tarea |

In [ ]:
DEPTH_GROUPS = [
    (list(range(0, 4)), 'Early (0-3)'),
    (list(range(4, 8)), 'Middle (4-7)'),
    (list(range(8, 12)), 'Late (8-11)'),
]

print(f'Grupos de profundidad: {len(DEPTH_GROUPS)}')
for layer_indices, label in DEPTH_GROUPS:
    names = filter_layer_names(all_target_names, layers=layer_indices)
    print(f'  {label}: layers {layer_indices} ({len(names)} capas)')
print(f'Rangos: {RANKS}')
print(f'Total modelos: {len(DEPTH_GROUPS) * len(RANKS)}')

In [ ]:
depth_results = []
depth_embeddings = {}

for layer_indices, depth_label in DEPTH_GROUPS:
    for rank in RANKS:
        print(f'\n{"="*50}')
        print(f'Depth: {depth_label}, Rank: {rank}')

        model, target_names = create_compressed_model(
            baseline_model, rank, layers=layer_indices,
        )
        model.to(device)

        metrics = evaluate_model(model)
        f1 = metrics['eval_f1_macro']
        ratio = get_compression_ratio(baseline_model, model)

        embs, _ = extract_cls_embeddings(model, sub_test)

        depth_results.append({
            'depth_group': depth_label,
            'layers': str(layer_indices),
            'rank': rank,
            'f1_macro': f1,
            'compression_ratio': ratio,
            'n_layers': len(target_names),
        })
        depth_embeddings[(depth_label, rank)] = embs

        print(f'  F1 macro: {f1:.4f}')
        print(f'  Compression ratio: {ratio:.2f}x')
        print(f'  Layers compressed: {len(target_names)}')

        del model
        if device == 'cuda':
            torch.cuda.empty_cache()

print(f'\n{"="*50}')
print(f'Modelos evaluados: {len(depth_results)}')

### 5.1 t-SNE y visualización por profundidad

Para cada nivel de rango, ejecutamos t-SNE conjunto sobre baseline + 3 grupos de profundidad
(4 modelos × 1500 puntos = 6,000 puntos por run).

In [ ]:
depth_tsne = {}
depth_silhouette = {}

for rank in RANKS:
    print(f'\nt-SNE para rank {rank}...')

    emb_dict = {'Baseline': baseline_embs}
    for layer_indices, depth_label in DEPTH_GROUPS:
        emb_dict[depth_label] = depth_embeddings[(depth_label, rank)]

    tsne_per_model, bounds = run_joint_tsne(emb_dict)
    depth_tsne[rank] = (tsne_per_model, bounds)

    sil_scores = compute_silhouette_scores(tsne_per_model, dominant_emotion, top_indices)
    for _, depth_label in DEPTH_GROUPS:
        depth_silhouette[(depth_label, rank)] = sil_scores[depth_label]
    depth_silhouette[('Baseline', rank)] = sil_scores['Baseline']

    print(f'  Silhouette - Baseline: {sil_scores["Baseline"]:.4f}')
    for _, depth_label in DEPTH_GROUPS:
        print(f'  Silhouette - {depth_label}: {sil_scores[depth_label]:.4f}')

In [ ]:
for rank in RANKS:
    tsne_per_model, bounds = depth_tsne[rank]

    f1_dict = {'Baseline': baseline_f1}
    for r in depth_results:
        if r['rank'] == rank:
            f1_dict[r['depth_group']] = r['f1_macro']

    plot_scatter_panel(
        tsne_per_model, bounds, dominant_emotion, top_indices,
        f1_scores=f1_dict, ncols=4,
        title=f'Compresión por profundidad — Rank {rank}\n(scatter de emociones dominantes, t-SNE conjunto)',
        save_path=os.path.join(results_dir, f'depth_scatter_rank{rank}.png'),
    )

In [ ]:
for rank in RANKS:
    tsne_per_model, bounds = depth_tsne[rank]

    f1_dict = {'Baseline': baseline_f1}
    for r in depth_results:
        if r['rank'] == rank:
            f1_dict[r['depth_group']] = r['f1_macro']

    plot_kde_panel(
        tsne_per_model, bounds, dominant_emotion, top_indices,
        f1_scores=f1_dict, ncols=4,
        title=f'Compresión por profundidad — Rank {rank}\n(densidad KDE de emociones, t-SNE conjunto)',
        save_path=os.path.join(results_dir, f'depth_kde_rank{rank}.png'),
    )

In [ ]:
print('Silhouette scores — Estudio por profundidad\n')

for rank in RANKS:
    print(f'{"="*45}')
    print(f'Rank {rank}')
    print(f'{"Modelo":<20s} | {"Silhouette":>10s}')
    print(f'{"-"*35}')
    print(f'{"Baseline":<20s} | {depth_silhouette[("Baseline", rank)]:>10.4f}')
    for _, depth_label in DEPTH_GROUPS:
        sil = depth_silhouette[(depth_label, rank)]
        print(f'{depth_label:<20s} | {sil:>10.4f}')
    print()

## 6. Dashboards resumen

Heatmaps comparativos de F1 macro y silhouette score para ambos estudios.

In [ ]:
# Preparar matrices para heatmaps — Componentes
comp_labels = [label for _, label in COMPONENTS]

f1_comp_matrix = np.zeros((len(COMPONENTS), len(RANKS)))
sil_comp_matrix = np.zeros((len(COMPONENTS), len(RANKS)))

for i, (_, comp_label) in enumerate(COMPONENTS):
    for j, rank in enumerate(RANKS):
        result = [r for r in comp_results
                  if r['component'] == comp_label and r['rank'] == rank][0]
        f1_comp_matrix[i, j] = result['f1_macro']
        sil_comp_matrix[i, j] = comp_silhouette[(comp_label, rank)]

# Heatmap F1 — Componente
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    f1_comp_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS],
    yticklabels=comp_labels,
    vmin=f1_comp_matrix.min() - 0.01, vmax=baseline_f1 + 0.005,
    ax=ax,
)
ax.set_xlabel('Rank SVD', fontsize=12)
ax.set_ylabel('Componente', fontsize=12)
ax.set_title(f'F1 macro por componente y rango\n(Baseline: {baseline_f1:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'heatmap_f1_component.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap Silhouette — Componente
baseline_sil_avg = np.mean([comp_silhouette[('Baseline', r)] for r in RANKS])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    sil_comp_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS],
    yticklabels=comp_labels,
    ax=ax,
)
ax.set_xlabel('Rank SVD', fontsize=12)
ax.set_ylabel('Componente', fontsize=12)
ax.set_title(f'Silhouette score por componente y rango\n(Baseline promedio: {baseline_sil_avg:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'heatmap_silhouette_component.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Preparar matrices para heatmaps — Profundidad
depth_labels = [label for _, label in DEPTH_GROUPS]

f1_depth_matrix = np.zeros((len(DEPTH_GROUPS), len(RANKS)))
sil_depth_matrix = np.zeros((len(DEPTH_GROUPS), len(RANKS)))

for i, (_, depth_label) in enumerate(DEPTH_GROUPS):
    for j, rank in enumerate(RANKS):
        result = [r for r in depth_results
                  if r['depth_group'] == depth_label and r['rank'] == rank][0]
        f1_depth_matrix[i, j] = result['f1_macro']
        sil_depth_matrix[i, j] = depth_silhouette[(depth_label, rank)]

# Heatmap F1 — Profundidad
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    f1_depth_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS],
    yticklabels=depth_labels,
    vmin=f1_depth_matrix.min() - 0.01, vmax=baseline_f1 + 0.005,
    ax=ax,
)
ax.set_xlabel('Rank SVD', fontsize=12)
ax.set_ylabel('Grupo de profundidad', fontsize=12)
ax.set_title(f'F1 macro por profundidad y rango\n(Baseline: {baseline_f1:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'heatmap_f1_depth.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap Silhouette — Profundidad
baseline_sil_depth_avg = np.mean([depth_silhouette[('Baseline', r)] for r in RANKS])

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    sil_depth_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
    xticklabels=[str(r) for r in RANKS],
    yticklabels=depth_labels,
    ax=ax,
)
ax.set_xlabel('Rank SVD', fontsize=12)
ax.set_ylabel('Grupo de profundidad', fontsize=12)
ax.set_title(f'Silhouette score por profundidad y rango\n(Baseline promedio: {baseline_sil_depth_avg:.4f})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'heatmap_silhouette_depth.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Dashboard combinado 2x2
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# F1 — Componente
sns.heatmap(f1_comp_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
            xticklabels=[str(r) for r in RANKS], yticklabels=comp_labels,
            vmin=f1_comp_matrix.min() - 0.01, vmax=baseline_f1 + 0.005,
            ax=axes[0, 0])
axes[0, 0].set_title(f'F1 macro — Componente\n(Baseline: {baseline_f1:.4f})', fontweight='bold')
axes[0, 0].set_xlabel('Rank SVD')
axes[0, 0].set_ylabel('Componente')

# Silhouette — Componente
sns.heatmap(sil_comp_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
            xticklabels=[str(r) for r in RANKS], yticklabels=comp_labels,
            ax=axes[0, 1])
axes[0, 1].set_title(f'Silhouette — Componente\n(Baseline: {baseline_sil_avg:.4f})', fontweight='bold')
axes[0, 1].set_xlabel('Rank SVD')
axes[0, 1].set_ylabel('')

# F1 — Profundidad
sns.heatmap(f1_depth_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
            xticklabels=[str(r) for r in RANKS], yticklabels=depth_labels,
            vmin=f1_depth_matrix.min() - 0.01, vmax=baseline_f1 + 0.005,
            ax=axes[1, 0])
axes[1, 0].set_title(f'F1 macro — Profundidad\n(Baseline: {baseline_f1:.4f})', fontweight='bold')
axes[1, 0].set_xlabel('Rank SVD')
axes[1, 0].set_ylabel('Grupo de profundidad')

# Silhouette — Profundidad
sns.heatmap(sil_depth_matrix, annot=True, fmt='.4f', cmap='RdYlGn',
            xticklabels=[str(r) for r in RANKS], yticklabels=depth_labels,
            ax=axes[1, 1])
axes[1, 1].set_title(f'Silhouette — Profundidad\n(Baseline: {baseline_sil_depth_avg:.4f})', fontweight='bold')
axes[1, 1].set_xlabel('Rank SVD')
axes[1, 1].set_ylabel('')

fig.suptitle('Dashboard: Impacto de la compresión SVD selectiva',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'dashboard_compression_analysis.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Barras agrupadas — Retención de F1
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

colors = ['#2ecc71', '#f39c12', '#e74c3c']

# Componentes
comp_retention = (f1_comp_matrix / baseline_f1) * 100
x = np.arange(len(comp_labels))
width = 0.25

for j, rank in enumerate(RANKS):
    axes[0].bar(x + j * width, comp_retention[:, j], width,
                label=f'Rank {rank}', color=colors[j], alpha=0.85)

axes[0].axhline(y=100, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Baseline')
axes[0].set_xlabel('Componente', fontsize=12)
axes[0].set_ylabel('Retención F1 (%)', fontsize=12)
axes[0].set_title('Retención de F1 por componente', fontsize=13, fontweight='bold')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(comp_labels, rotation=45, ha='right')
axes[0].legend()
axes[0].set_ylim(0, 105)

# Profundidad
depth_retention = (f1_depth_matrix / baseline_f1) * 100
x_d = np.arange(len(depth_labels))

for j, rank in enumerate(RANKS):
    axes[1].bar(x_d + j * width, depth_retention[:, j], width,
                label=f'Rank {rank}', color=colors[j], alpha=0.85)

axes[1].axhline(y=100, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Baseline')
axes[1].set_xlabel('Grupo de profundidad', fontsize=12)
axes[1].set_ylabel('Retención F1 (%)', fontsize=12)
axes[1].set_title('Retención de F1 por profundidad', fontsize=13, fontweight='bold')
axes[1].set_xticks(x_d + width)
axes[1].set_xticklabels(depth_labels, rotation=45, ha='right')
axes[1].legend()
axes[1].set_ylim(0, 105)

fig.suptitle('Retención de rendimiento bajo compresión SVD selectiva',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'barchart_f1_retention.png'),
            dpi=200, bbox_inches='tight')
plt.show()

## 7. Guardar resultados

In [ ]:
# Exportar resultados de componentes
df_comp = pd.DataFrame(comp_results)
df_comp['f1_retention'] = df_comp['f1_macro'] / baseline_f1 * 100
df_comp['silhouette'] = df_comp.apply(
    lambda row: comp_silhouette.get((row['component'], row['rank']), np.nan), axis=1
)
df_comp = df_comp.drop(columns=['component_key'])

comp_csv_path = os.path.join(results_dir, 'analysis_component_results.csv')
df_comp.to_csv(comp_csv_path, index=False)
print(f'Guardado: {comp_csv_path}')
print(f'  Filas: {len(df_comp)}')
print(df_comp.to_string(index=False))

print()

# Exportar resultados de profundidad
df_depth = pd.DataFrame(depth_results)
df_depth['f1_retention'] = df_depth['f1_macro'] / baseline_f1 * 100
df_depth['silhouette'] = df_depth.apply(
    lambda row: depth_silhouette.get((row['depth_group'], row['rank']), np.nan), axis=1
)

depth_csv_path = os.path.join(results_dir, 'analysis_depth_results.csv')
df_depth.to_csv(depth_csv_path, index=False)
print(f'Guardado: {depth_csv_path}')
print(f'  Filas: {len(df_depth)}')
print(df_depth.to_string(index=False))

In [ ]:
figures = [
    'comp_scatter_rank256.png', 'comp_scatter_rank128.png', 'comp_scatter_rank64.png',
    'comp_kde_rank256.png', 'comp_kde_rank128.png', 'comp_kde_rank64.png',
    'depth_scatter_rank256.png', 'depth_scatter_rank128.png', 'depth_scatter_rank64.png',
    'depth_kde_rank256.png', 'depth_kde_rank128.png', 'depth_kde_rank64.png',
    'heatmap_f1_component.png', 'heatmap_silhouette_component.png',
    'heatmap_f1_depth.png', 'heatmap_silhouette_depth.png',
    'dashboard_compression_analysis.png', 'barchart_f1_retention.png',
]

print('=' * 60)
print('RESUMEN — Fase 4: Análisis por componente y profundidad')
print('=' * 60)
print(f'\nBaseline F1 macro: {baseline_f1:.4f}')
print(f'Muestras para t-SNE: {N_SAMPLES}')
print(f'Modelos evaluados: {1 + len(comp_results) + len(depth_results)}'
      f' (1 baseline + {len(comp_results)} componente + {len(depth_results)} profundidad)')
print(f'Runs de t-SNE: 6 (3 componente + 3 profundidad)')
print(f'\nFiguras generadas en results/:')
for fig_name in figures:
    path = os.path.join(results_dir, fig_name)
    exists = 'ok' if os.path.exists(path) else 'MISSING'
    print(f'  [{exists}] {fig_name}')
print(f'\nCSVs:')
print(f'  analysis_component_results.csv ({len(df_comp)} filas)')
print(f'  analysis_depth_results.csv ({len(df_depth)} filas)')